# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export — Query ADS API, resolve bibcodes to metadata
2. Translation — Detect languages, translate non-English titles/abstracts
3. Tokenization — Create full_text, tokenize with spaCy
4. AND — Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation — BERTopic + datamapplot + cluster removal
6. Citation Networks — Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Verify
print(f"Project root: {paths['project_root']}")
print(f"Data dir:     {paths['data']}")

Project root: c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline
Data dir:     c:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\data


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

In [2]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

# Example: Einstein's GR papers + citation expansion + free-text search
Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"  # Einstein GR library
Set_B = f"citations({Set_A}) AND year:1915-2000"   # 1-hop citations
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"  # 2-hop
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"  # Treder library

gr_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gr_terms)}) AND year:1911-2000"
QUERY = f"author:\"Treder, H*\""   # f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

# Set to False to load the latest cached results instead of re-fetching
FORCE_REFRESH = False

### 1.2 Execute Search

In [3]:
from ads_bib.search import search_ads, save_search_results
from ads_bib._utils.io import load_pickle

latest = paths["raw"] / "search_results_latest.pkl"

if FORCE_REFRESH or not latest.exists():
    bibcodes, references, esources, fulltext_urls = search_ads(QUERY, ADS_TOKEN)
    save_search_results((bibcodes, references, esources, fulltext_urls), paths["raw"])
else:
    print(f"Loading cached results: {latest.name}")
    bibcodes, references, esources, fulltext_urls = load_pickle(latest)

print(f"\nBibcodes:        {len(bibcodes):,}")
print(f"Unique refs:     {len(set(r for rl in references for r in rl)):,}")
print(f"Total refs:      {sum(len(rl) for rl in references):,}")

Loading cached results: search_results_latest.pkl

Bibcodes:        382
Unique refs:     499
Total refs:      796


### 1.3 Export & Resolve Metadata

In [4]:
from ads_bib.export import resolve_dataset
from ads_bib._utils.io import save_json_lines, load_json_lines

pubs_path = paths["raw"] / "publications.json"
refs_path = paths["raw"] / "references.json"

if FORCE_REFRESH or not pubs_path.exists():
    publications, refs = resolve_dataset(
        bibcodes, references, esources, fulltext_urls, ADS_TOKEN
    )
    save_json_lines(publications, pubs_path)
    save_json_lines(refs, refs_path)
else:
    print("Loading cached exports ...")
    publications = load_json_lines(pubs_path)
    refs = load_json_lines(refs_path)

print(f"\nPublications: {len(publications):,}")
print(f"References:   {len(refs):,}")
publications.info()

Loading cached exports ...

Publications: 382
References:   499
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382 entries, 0 to 381
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Bibcode               382 non-null    object
 1   Author                382 non-null    object
 2   Title                 382 non-null    object
 3   Year                  382 non-null    int64 
 4   Journal               382 non-null    object
 5   Journal Abbreviation  382 non-null    object
 6   Issue                 382 non-null    object
 7   Volume                382 non-null    object
 8   First Page            382 non-null    object
 9   Last Page             382 non-null    object
 10  Abstract              382 non-null    object
 11  Keywords              382 non-null    object
 12  DOI                   382 non-null    object
 13  Affiliation           382 non-null    object
 14  Category              382 

In [5]:
publications.head(10)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,1971grun.book.....T,Treder HJ,Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,,,,AA(-),book,52,[],None
1,1957AnP...454..369T,Treder H,Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,"Wie Infeld 1950 gezeigt hat, ergibt sich aus d...",,10.1002/andp.19574540614,AA(-),article,34,[1953PhRv...92.1567C],None
2,1967AnP...475..194T,Treder HJ,Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,A theory of the gravitation field is proposed ...,,10.1002/andp.19674750308,AA(-),article,38,"[1916AnP...354..769E, 1940PhRv...57..147R, 195...",None
3,1983AN....304..145T,Treder HJ,The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,The evolution and fundamental questions of phy...,"Astronomical Models, Cosmology, Unified Field ...",10.1002/asna.2113040402,"AA(German Academy of Sciences, Potsdam)",article,1,[1950sts..book.....S],https://ui.adsabs.harvard.edu/link_gateway/198...
4,1972drdt.book.....T,Treder HJ,Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,,,,AA(-),book,31,[],None
5,1997GReGr..29..455V,"von Borzeszkowski HH, Treder HJ",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,"According to Poincaré, only the ``epistemologi...",,10.1023/A:1018830631884,AA(-); AB(-),article,21,"[1950sts..book.....S, 1971GReGr...2..313T, 199...",None
6,1975AnP...487..383T,Treder HJ,Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,,,10.1002/andp.19754870509,AA(-),article,26,"[1963ForPh..11...81T, 1973AnP...485..229T, 197...",None
7,1981AN....302..275T,"Treder HJ, Jackisch G",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,,,10.1002/asna.2103020603,AA(-); AB(-),article,6,"[1970AnP...480..315T, 1974AnP...486..325T]",https://ui.adsabs.harvard.edu/link_gateway/198...
8,1988mqg..book.....V,"von Borzeszkowski HH, Treder HJ",The meaning of quantum gravity.,1988,The meaning of quantum gravity.. H.-H. von Bor...,mqg..book,,20,,,Should the general theory of relativity be qua...,,,AA(-); AB(-),book,20,[],None
9,1980AnP...492..250T,Treder HJ,Einstein's Hermitian theory of relativity as u...,1980,Annalen der Physik,AnP,,492,250,258,,"Relativistic Astrophysics, Gravitation Theory,...",10.1002/andp.19804920403,AA(-),article,19,"[1957AnP...454..369T, 1975AnP...487..375T]",None


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

In [ ]:
# --- TRANSLATION CONFIGURATION ---
# Provider options: 'openrouter' (API) or 'huggingface' (local, with/without CUDA)

TRANSLATION_PROVIDER = "openrouter"  # or "huggingface"
TRANSLATION_MODEL = "google/gemini-3-flash-preview"  # OpenRouter model, or "google/translategemma-4b-it" for HF
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 5  # parallel workers for API-based translation

# Path to fasttext language ID model (download from https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin)
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"

### 2.2 Language Detection

In [7]:
from ads_bib.translate import detect_languages

print("=== Publications ===")
publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

print("\n=== References ===")
refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)

=== Publications ===
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected

=== References ===
  Title: 185 non-English entries detected
  Abstract: 36 non-English entries detected


### 2.3 Translate Non-English Entries

In [8]:
from ads_bib.translate import translate_dataframe

print("=== Translating Publications ===")
publications, cost_pubs = translate_dataframe(
    publications, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
)
print(f"  Cost: {cost_pubs}")

print("\n=== Translating References ===")
refs, cost_refs = translate_dataframe(
    refs, ["Title", "Abstract"],
    provider=TRANSLATION_PROVIDER,
    model=TRANSLATION_MODEL,
    api_key=TRANSLATION_API_KEY,
    max_workers=TRANSLATION_MAX_WORKERS,
)
print(f"  Cost: {cost_refs}")

=== Translating Publications ===
  Title: translating 183 entries with openrouter/gemini-3-flash-preview ...


  Title: 100%|██████████| 183/183 [00:12<00:00, 14.22it/s]


  Title: 183 translations failed
  Abstract: translating 35 entries with openrouter/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 35/35 [00:01<00:00, 24.92it/s]


  Abstract: 35 translations failed
  Cost: {'prompt_tokens': 0, 'completion_tokens': 0, 'provider': 'openrouter', 'model': 'gemini-3-flash-preview'}

=== Translating References ===
  Title: translating 185 entries with openrouter/gemini-3-flash-preview ...


  Title: 100%|██████████| 185/185 [00:04<00:00, 41.98it/s]


  Title: 185 translations failed
  Abstract: translating 36 entries with openrouter/gemini-3-flash-preview ...


  Abstract: 100%|██████████| 36/36 [00:00<00:00, 48.39it/s]

  Abstract: 36 translations failed
  Cost: {'prompt_tokens': 0, 'completion_tokens': 0, 'provider': 'openrouter', 'model': 'gemini-3-flash-preview'}


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize with spaCy.

In [9]:
from ads_bib.tokenize import tokenize_texts

publications = tokenize_texts(publications)
refs = tokenize_texts(refs)

c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_lg' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Tokenizing 382 documents with en_core_web_lg ...
  Done.


c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'en_core_web_lg' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Tokenizing 499 documents with en_core_web_lg ...
  Done.


### Checkpoint: Save Phase 1-3 Results

In [10]:
from ads_bib._utils.io import save_json_lines

save_json_lines(publications, paths["processed"] / "publications_translated_tokenized.json")
save_json_lines(refs, paths["processed"] / "references_translated_tokenized.json")
print("Checkpoint saved.")

Checkpoint saved.


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [11]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

print("AND step skipped (placeholder). Continuing without disambiguation.")

AND step skipped (placeholder). Continuing without disambiguation.


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

In [12]:
# --- EMBEDDING CONFIGURATION ---
# Providers: 'local', 'huggingface_api', 'openrouter'

EMBEDDING_PROVIDER = "openrouter"
EMBEDDING_MODEL = "google/gemini-embedding-001"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")  # only needed for API providers

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None

### 5.2 Compute Embeddings

In [13]:
from ads_bib.topic_model import compute_embeddings

df = publications.copy()
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    print(f"SAMPLING: {len(df):,} documents")

documents = df["full_text"].tolist()

embeddings = compute_embeddings(
    documents,
    provider=EMBEDDING_PROVIDER,
    model=EMBEDDING_MODEL,
    cache_dir=paths["embeddings_cache"],
    api_key=EMBEDDING_API_KEY,
)
print(f"Embeddings shape: {embeddings.shape}")

  Computing embeddings with openrouter/google/gemini-embedding-001 ...


Embedding (OpenRouter): 100%|██████████| 6/6 [00:17<00:00,  2.96s/it]


  Saved: embeddings_openrouter_google_gemini-embedding-001.npz
Embeddings shape: (382, 3072)


### 5.3 Dimensionality Reduction Configuration

In [14]:
# --- DIMENSIONALITY REDUCTION ---
# Methods: 'pacmap' (fast, good) or 'umap' (slower, density-preserving with densmap)

DIM_REDUCTION_METHOD = "pacmap"

# PaCMAP parameters
PARAMS_5D = dict(n_neighbors=60, MN_ratio=0.5, FP_ratio=1.0, random_state=42)
PARAMS_2D = dict(n_neighbors=60, MN_ratio=0.5, FP_ratio=1.0, random_state=42)

# Cache suffix for uniqueness
model_safe = EMBEDDING_MODEL.replace('/', '_')
CACHE_SUFFIX = f"{EMBEDDING_PROVIDER}_{model_safe}_{DIM_REDUCTION_METHOD}"

In [15]:
from ads_bib.topic_model import reduce_dimensions

reduced_5d, reduced_2d = reduce_dimensions(
    embeddings,
    method=DIM_REDUCTION_METHOD,
    params_5d=PARAMS_5D,
    params_2d=PARAMS_2D,
    cache_dir=paths["dim_reduction_cache"],
    cache_suffix=CACHE_SUFFIX,
)

  Computing 5d_openrouter_google_gemini-embedding-001_pacmap with PACMAP ...


Note: `n_components != 2` have not been thoroughly tested.


Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=60, n_MN=30, n_FP=60, distance=euclidean, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=42)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((22920, 2), (11460, 2), (22920, 2))
Initial Loss: 16773.466796875
Iteration:   10, Loss: 18039.556641
Iteration:   20, Loss: 15166.378906
Iteration:   30, Loss: 14516.935547
Iteration:   40, Loss: 14134.623047
Iteration:   50, Loss: 13826.110352
Iteration:   60, Loss: 13549.056641
Iteration:   70, Loss: 13273.548828
Iteration:   80, Loss: 12990.171875
Iteration:   90, Loss: 12693.007812
Iteration:  100, Loss: 12377.171875
Iteration:  110, Loss: 15591.467773
Iteration:  120, Loss: 15548.587891
Iteration:  130, Loss: 15530.492188
Iteration:  140, Loss: 15525.212891
Iteration:  150, Loss: 15522.830078
Iteration:  160, Loss: 15522.298828
Iteration:  170, Loss: 15521.775391
Itera

Iteration:  450, Loss: 8308.255859
Elapsed time: 0.68s
  Saved: reduced_5d_openrouter_google_gemini-embedding-001_pacmap.npy
  Computing 2d_openrouter_google_gemini-embedding-001_pacmap with PACMAP ...
Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=60, n_MN=30, n_FP=60, distance=euclidean, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=42)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((22920, 2), (11460, 2), (22920, 2))
Initial Loss: 16773.435546875
Iteration:   10, Loss: 16018.837891
Iteration:   20, Loss: 14900.164062
Iteration:   30, Loss: 14449.281250
Iteration:   40, Loss: 14142.614258
Iteration:   50, Loss: 13876.910156
Iteration:   60, Loss: 13615.898438
Iteration:   70, Loss: 13349.270508
Iteration:   80, Loss: 13071.087891
Iteration:   90, Loss: 12776.952148
Iteration:  100, Loss: 12462.395508
Iteration:  110, Loss: 15621.091797
Iteration:  120, Lo

### 5.4 Clustering Configuration

**Adjust `min_cluster_size` based on your dataset size!**
- ~180k docs → `min_cluster_size=180` (~0.1%)
- ~50k docs → `min_cluster_size=100`
- ~5k docs → `min_cluster_size=25`
- Testing (200 docs) → `min_cluster_size=10`

In [16]:
# --- CLUSTERING CONFIGURATION ---
CLUSTERING_METHOD = "fast_hdbscan"  # or 'hdbscan'

n_docs = len(df)
MIN_CLUSTER_SIZE = max(10, int(n_docs * 0.001))  # ~0.1% of dataset
print(f"Dataset: {n_docs:,} documents → min_cluster_size={MIN_CLUSTER_SIZE}")

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.02,
)

Dataset: 382 documents → min_cluster_size=10


In [17]:
from ads_bib.topic_model import cluster_documents

clusters = cluster_documents(reduced_5d, method=CLUSTERING_METHOD, params=CLUSTER_PARAMS)

  FAST_HDBSCAN: 6 topics, 160 outliers (41.9%)


### 5.5 BERTopic Configuration

In [20]:
# --- BERTOPIC / LLM LABELING ---
# LLM Providers: 'local', 'huggingface_api', 'openrouter'

LLM_PROVIDER = "openrouter"
LLM_MODEL = "google/gemini-3-flash-preview"  # or an OpenRouter model
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Representation pipeline (run before LLM)
PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

# min_df for CountVectorizer: use 1 for small samples, 2+ for full data
MIN_DF = 1 if (SAMPLE_SIZE and SAMPLE_SIZE < 1000) else 2

In [21]:
from ads_bib.topic_model import fit_bertopic, reduce_outliers, build_topic_dataframe
import numpy as np

topic_model = fit_bertopic(
    documents, reduced_5d, clusters,
    llm_provider=LLM_PROVIDER,
    llm_model=LLM_MODEL,
    pipeline_models=PIPELINE_MODELS,
    parallel_models=PARALLEL_MODELS,
    min_df=MIN_DF,
    api_key=LLM_API_KEY,
)

topics = np.array(topic_model.topics_)
topics = reduce_outliers(topic_model, documents, topics, reduced_5d, threshold=0.8)

df = build_topic_dataframe(df, topic_model, topics, reduced_2d, embeddings)
topic_model.get_topic_info()

2026-02-18 17:16:34,332 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-18 17:16:34,333 - BERTopic - Dimensionality - Completed ✓
2026-02-18 17:16:34,334 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-18 17:16:34,339 - BERTopic - Cluster - Completed ✓
2026-02-18 17:16:34,342 - BERTopic - Representation - Fine-tuning topics using representation models.


Fitting BERTopic (LLM: openrouter/google/gemini-3-flash-preview) ...


100%|██████████| 5/5 [00:22<00:00,  4.48s/it]
2026-02-18 17:17:07,127 - BERTopic - Representation - Completed ✓
2026-02-18 17:17:07,242 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


  Outliers: 186 → 116


100%|██████████| 5/5 [00:16<00:00,  3.25s/it]


,Topic,Count,Name,CustomName,Representation,MMR,POS,KeyBERT,Representative_Docs
0,-1,116,-1_Classical and Quantum General Relativity Th...,Outlier Topic,[Classical and Quantum General Relativity Theo...,"[gravitation, vorticity, relativistic, einstei...","[der, gravitation, und, theory, die, vorticity...","[theory gravitation, gravitationstheorien, der...",[Mach-Einstein doctrine and general relativity...
1,0,124,0_Unified Field Theories and General Relativity,Unified Field Theories and General Relativity,[Unified Field Theories and General Relativity],"[einstein, general relativity, gravitation, fi...","[field, equations, theory, general, der, relat...","[general relativity, einstein equations, gener...",[Hamiltonian dynamics of purely affine fields ...
2,1,88,1_History of Geophysics and Cosmogony,History of Geophysics and Cosmogony,[History of Geophysics and Cosmogony],"[solar, kosmologie, kosmos, cosmology, physics...","[und, der, solar, zum, cosmology, geophysical,...","[kosmologie und, kosmologie, cosmology, physik...","[Galaxienbewegung und Frühphase des Kosmos. , ..."
3,2,28,2_Mach's Principle and the Relativity of Inertia,Mach's Principle and the Relativity of Inertia,[Mach's Principle and the Relativity of Inertia],"[mach einstein doctrine, doctrine, einstein do...","[principle, doctrine, mechanics, matter, inert...","[mach einstein doctrine, einstein doctrine, co...",[Remarks on anisotropy of inertia in an anisot...
4,3,26,3_Foundations of Physics and Elementary Constants,Foundations of Physics and Elementary Constants,[Foundations of Physics and Elementary Constants],"[quantum, planck, constants, elementary consta...","[die, der, physics, und, quantum, elementary, ...","[planck, der physik und, physik und, quantum m...",[Die Elementarkonstanten und die Mathematik. D...


### 5.6 Visualize Topics

In [22]:
from ads_bib.visualize import create_topic_map

plot = create_topic_map(
    df,
    label_column="Name",
    title="ADS Bibliometric Map",
    subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
    dark_mode=True,
    output_path=paths["output"] / "topic_map.html",
)
# plot  # uncomment to display inline

ValueError: The value of polygon_alpha was too low, and no boundary was formed. Try increasing polygon_alpha.

### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
from ads_bib.curate import get_cluster_summary, remove_clusters

get_cluster_summary(df)

In [ ]:
# Remove unwanted clusters (set IDs based on the summary above)
CLUSTERS_TO_REMOVE = []  # e.g. [3, 7, -1]

if CLUSTERS_TO_REMOVE:
    df = remove_clusters(df, CLUSTERS_TO_REMOVE)

### Checkpoint: Save Curated Dataset

In [ ]:
from ads_bib._utils.io import save_parquet

# Ensure References is list type for Parquet
df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

save_parquet(df, paths["processed"] / "curated_dataset.parquet")
print(f"Curated dataset: {len(df):,} documents")

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to SQLite (Gephi), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

In [ ]:
# --- CITATION CONFIGURATION ---
METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None  # e.g. ['Treder', 'Kreisel'] or None for all
OUTPUT_FORMAT = "sqlite"  # 'sqlite', 'csv', or 'both'

### 6.2 Build Node Table & Compute Networks

In [ ]:
from ads_bib.citations import build_all_nodes, process_all_citations, export_wos_format

# Reload processed data if needed
all_nodes = build_all_nodes(publications, refs)

results = process_all_citations(
    bibcodes, references, publications, refs, all_nodes,
    metrics=METRICS,
    min_counts=MIN_COUNTS,
    authors_filter=AUTHORS_FILTER,
    output_format=OUTPUT_FORMAT,
    output_dir=paths["output"],
)

### 6.3 Export Web of Science Format

In [ ]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=paths["output"] / f"wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [ ]:
import os

print("="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Publications:     {len(publications):,}")
print(f"References:       {len(refs):,}")
print(f"Curated dataset:  {len(df):,}")
print(f"Topics found:     {df['Cluster'].nunique()}")
print()
print("Output files:")
for root, dirs, files in os.walk(paths["output"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f"  {fpath.relative_to(paths['output'])} ({size_mb:.1f} MB)")